# Dataset Overview

This notebook provides an overview of the DadaGP dataset:
- Count of GP files and token files
- Dataset statistics
- Sample file analysis (without loading all files)

In [ ]:
import os
from pathlib import Path
from collections import Counter
import random

print("Current working directory:", Path.cwd())
print("Parent directory:", Path.cwd().parent)

## 1. Count Files by Type

In [ ]:
def count_files_by_extension(root_dir, show_progress=False):
    """Count files by extension without loading them."""
    files_by_ext = Counter()
    all_files = []
    
    print(f"Scanning {root_dir}...")
    for root, dirs, files in os.walk(root_dir):
        for file in files:
            ext = Path(file).suffix.lower()
            if ext:  # Has extension
                files_by_ext[ext] += 1
                all_files.append(Path(root) / file)
    
    return files_by_ext, all_files

# Count DadaGP files
dadagp_dir = '../DadaGP-v1.1'
if Path(dadagp_dir).exists():
    print("="*60)
    print("DadaGP-v1.1 DATASET")
    print("="*60)
    
    counts, all_files = count_files_by_extension(dadagp_dir)
    
    # Show file counts
    total = sum(counts.values())
    print(f"\nTotal files: {total:,}")
    print("\nTop 10 file types:")
    for ext, count in counts.most_common(10):
        print(f"  {ext:15s}: {count:6,} files ({count/total*100:5.1f}%)")
    
    # Count token files specifically
    token_files = [f for f in all_files if f.name.endswith('.tokens.txt')]
    gp_files = [f for f in all_files if f.suffix.lower() in ['.gp3', '.gp4', '.gp5', '.gpx']]
    
    print(f"\nGuitar Pro files: {len(gp_files):,}")
    print(f"Token files (.tokens.txt): {len(token_files):,}")
    
else:
    print(f"Directory not found: {dadagp_dir}")
    token_files = []
    gp_files = []

## 2. Sample Token Files

Select a few random token files for analysis (without loading all).

In [ ]:
if token_files:
    # Sample 5 random files
    sample_size = min(5, len(token_files))
    sample_files = random.sample(token_files, sample_size)
    
    print(f"Selected {sample_size} random token files for analysis:")
    print("="*60)
    for i, f in enumerate(sample_files, 1):
        rel_path = f.relative_to(Path(dadagp_dir))
        print(f"{i}. {rel_path}")
        
        # Quick file stats
        size_kb = f.stat().st_size / 1024
        line_count = sum(1 for _ in open(f))
        print(f"   Size: {size_kb:.1f} KB, Lines: {line_count:,}")
        print()
else:
    print("No token files found")

## 3. Parse First Sample File (Raw Tokens)

In [ ]:
if token_files:
    sample_file = sample_files[0]
    print(f"Analyzing: {sample_file.name}")
    print("="*60)
    
    # Read raw tokens
    with open(sample_file, 'r') as f:
        lines = [line.strip() for line in f if line.strip()]
    
    print(f"Total lines: {len(lines):,}")
    
    # Count token types
    token_types = Counter()
    for line in lines:
        if ':' in line:
            token_type = line.split(':')[0]
            token_types[token_type] += 1
        else:
            token_types[line] += 1
    
    print("\nToken type distribution:")
    for token_type, count in token_types.most_common(15):
        print(f"  {token_type:20s}: {count:5,} ({count/len(lines)*100:5.1f}%)")
    
    print("\nFirst 30 tokens:")
    print("-"*60)
    for i, line in enumerate(lines[:30], 1):
        print(f"{i:3d}. {line}")

## 4. Analyze Token Patterns

In [ ]:
if token_files:
    # Count specific token patterns
    note_tokens = [line for line in lines if ':note:' in line]
    wait_tokens = [line for line in lines if line.startswith('wait:')]
    measure_tokens = [line for line in lines if line == 'new_measure']
    effect_tokens = [line for line in lines if line.startswith(('nfx:', 'bfx:'))]
    
    print("Token category counts:")
    print(f"  Notes:       {len(note_tokens):5,}")
    print(f"  Waits:       {len(wait_tokens):5,}")
    print(f"  Measures:    {len(measure_tokens):5,}")
    print(f"  Effects:     {len(effect_tokens):5,}")
    
    # Analyze note tokens
    if note_tokens:
        print("\nNote token analysis:")
        
        # Extract string and fret info
        strings = []
        frets = []
        instruments = []
        
        for note in note_tokens:
            parts = note.split(':')
            if len(parts) >= 4:
                instruments.append(parts[0])
                # Parse string (s1, s2, etc.)
                string_str = parts[2]
                if string_str.startswith('s'):
                    strings.append(int(string_str[1:]))
                # Parse fret (f0, f1, etc.)
                fret_str = parts[3]
                if fret_str.startswith('f'):
                    frets.append(int(fret_str[1:]))
        
        # Show distributions
        print(f"  Instruments: {Counter(instruments).most_common()}")
        
        if strings:
            string_counts = Counter(strings)
            print(f"  Strings (1-6):")
            for s in sorted(string_counts.keys()):
                print(f"    String {s}: {string_counts[s]:4,} notes ({string_counts[s]/len(strings)*100:5.1f}%)")
        
        if frets:
            fret_counts = Counter(frets)
            print(f"  Frets (top 10):")
            for fret, count in fret_counts.most_common(10):
                print(f"    Fret {fret:2d}: {count:4,} notes ({count/len(frets)*100:5.1f}%)")

## 5. Summary Statistics Across Multiple Files

In [ ]:
if token_files:
    print("Analyzing multiple files for aggregate statistics...")
    print("="*60)
    
    # Analyze up to 20 random files
    analysis_files = random.sample(token_files, min(20, len(token_files)))
    
    total_notes = 0
    total_measures = 0
    all_strings = []
    all_frets = []
    file_lengths = []
    
    for f in analysis_files:
        with open(f, 'r') as file:
            lines = [line.strip() for line in file if line.strip()]
            file_lengths.append(len(lines))
            
            for line in lines:
                if ':note:' in line:
                    total_notes += 1
                    parts = line.split(':')
                    if len(parts) >= 4:
                        # Parse string and fret
                        string_str = parts[2]
                        fret_str = parts[3]
                        if string_str.startswith('s'):
                            all_strings.append(int(string_str[1:]))
                        if fret_str.startswith('f'):
                            all_frets.append(int(fret_str[1:]))
                elif line == 'new_measure':
                    total_measures += 1
    
    print(f"Analyzed {len(analysis_files)} files")
    print(f"\nAggregate statistics:")
    print(f"  Total notes: {total_notes:,}")
    print(f"  Total measures: {total_measures:,}")
    print(f"  Avg notes per file: {total_notes/len(analysis_files):.1f}")
    print(f"  Avg measures per file: {total_measures/len(analysis_files):.1f}")
    print(f"  Avg tokens per file: {sum(file_lengths)/len(file_lengths):.1f}")
    print(f"  File size range: {min(file_lengths)} - {max(file_lengths)} tokens")
    
    if all_strings:
        print(f"\nString usage across all files:")
        string_counts = Counter(all_strings)
        for s in sorted(string_counts.keys()):
            print(f"  String {s}: {string_counts[s]:6,} notes ({string_counts[s]/len(all_strings)*100:5.1f}%)")
    
    if all_frets:
        print(f"\nMost common frets across all files:")
        fret_counts = Counter(all_frets)
        for fret, count in fret_counts.most_common(15):
            print(f"  Fret {fret:2d}: {count:6,} notes ({count/len(all_frets)*100:5.1f}%)")